# Gait Classifier

## 1. Load extracted features

In [1]:
from ambient.classification.features import GaitFeatureVector
from ambient.utils.features_utils import load_features

FILENAME="extracted_features.pkl"
DIRECTORY="features"

all_features, condition_counts = load_features(filename=FILENAME, directory=DIRECTORY)
print(f"\n{'='*60}")
print(f"Total features loaded: {len(all_features)}")
print(f"\nCondition distribution:")
for condition, count in sorted(condition_counts.items()):
     print(f"  {condition:15s}: {count:3d} samples")

✓ Features loaded successfully from: features/extracted_features.pkl
  Saved on: 2026-01-24T00:29:12.626096
  Total features: 68
  Conditions: ['stroke', 'myopathic', 'parkinsons', 'normal', 'cerebralpalsy']

Total features loaded: 68

Condition distribution:
  cerebralpalsy  :  15 samples
  myopathic      :  20 samples
  normal         :  12 samples
  parkinsons     :   9 samples
  stroke         :  12 samples


In [2]:
type(all_features)

list

## 2. Splitting Training vs Testing datasets

In [3]:
import numpy as np

In [4]:
def get_train_test(all_features: list, train_portion: float = 0.7):
    np.random.seed(42)
    split_idx = int( train_portion * len(all_features) )

    indices = np.random.permutation(len(all_features))

    train_indices = indices[:split_idx]
    test_indices = indices[split_idx:]

    train_features = [all_features[i] for i in train_indices]
    test_features = [all_features[i] for i in test_indices]
    
    return train_features, test_features

In [5]:
all_features = [f for f in all_features if f is not None]

In [6]:
train_features, test_features = get_train_test(all_features, 0.6)

len(train_features), len(test_features), len(all_features)

(40, 27, 67)

## 3. Classification with KNN

In [7]:
from ambient.classification.knn_classifier import (
    KNNGaitClassifier,
    KNNClassifierConfig
)

In [8]:
config = KNNClassifierConfig(
    n_neighbors=3,
    weights="distance",
    metric="euclidean",
    normalize_features=True
)

In [9]:
knn_classifier = KNNGaitClassifier(config=config)

2026-01-27 21:24:01.777 | INFO     | ambient.classification.knn_classifier:__init__:87 - KNN classifier initialized with k=3


### Training

In [10]:
knn_metrics = knn_classifier.train(
    features=train_features,
    validate=True,
    auto_remove_invalid=True
)

AttributeError: 'NoneType' object has no attribute 'items'

### Evaluation

In [ ]:
eval_metrics = knn_classifier.evaluate(train_features)
print(f"Eval accuracy: {eval_metrics['accuracy']}")

In [ ]:
eval_metrics = knn_classifier.evaluate(test_features=test_features)
print(f"Eval accuracy: {eval_metrics['accuracy']}")

In [ ]:
report = eval_metrics['classification_report']
for class_name in eval_metrics['classes']:
    cm = report[class_name]
    print(f"\n{class_name}")
    print(f"  F1-Score : {cm['f1-score']:.3f}")
    print(f"  Precision: {cm['precision']:.3f}")
    print(f"  Recall   : {cm['recall']:.3f}")    

### Evaluation Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(
    eval_metrics['confusion_matrix'],
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=eval_metrics['classes'],
    yticklabels=eval_metrics['classes']
)
plt.title('KNN Classifier - Confusion Matrix', fontsize=15)
plt.ylabel('True Class Label', fontsize=12)
plt.xlabel('Precited Class Label', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Classification with Decision Tree

In [ ]:
from ambient.classification.decisiontree_classifier import(
    DecisionTreeGaitClassifier,
    DecisionTreeClassifierConfig
)

In [ ]:
dt_config = DecisionTreeClassifierConfig(
    criterion='gini',
    max_depth=5,
    min_samples_split=3,
    min_samples_leaf=2,
    class_weight='balanced',
    normalize_features=True,
    random_state=42
)

dt_classifier = DecisionTreeGaitClassifier(dt_config)

### Training

In [ ]:
%%time 

dt_metrics = dt_classifier.train(
    features=train_features,
    validate=True,
    auto_remove_invalid=True
)

print(f"training accuracy: {dt_metrics['train_accuracy']}")

### Evaluation

In [ ]:
eval_metrics = dt_classifier.evaluate(train_features)
print(f"Eval accuracy: {eval_metrics['accuracy']}")

In [ ]:
eval_metrics = dt_classifier.evaluate(test_features)
print(f"Eval accuracy: {eval_metrics['accuracy']}")

In [ ]:
report = eval_metrics['classification_report']
for class_name in eval_metrics['classes']:
    cm = report[class_name]
    print(f"\n{class_name}")
    print(f"  F1-Score : {cm['f1-score']:.3f}")
    print(f"  Precision: {cm['precision']:.3f}")
    print(f"  Recall   : {cm['recall']:.3f}")    

In [ ]:
if 'feature_importances' in dt_metrics:
    print("Top 5 Most Important Features:")
    for feat in dt_metrics['feature_importances'][:5]:
        print(f"    {feat['rank']}. {feat['feature']:20s}: {feat['importance']:.3f}")

### Visualize Decision Tree

In [ ]:
from sklearn.tree import plot_tree, export_text
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Get the underlying sklearn model
sklearn_model = dt_classifier.model

# Method 1: Try to get feature names from GaitFeatureVector
try:
    feature_names = GaitFeatureVector.get_feature_names()
    print(f"✓ Got feature names from GaitFeatureVector: {len(feature_names)} features")
except AttributeError:
    print("GaitFeatureVector.get_feature_names() not available, trying alternatives...")
    
    # Method 2: Extract from sample feature vector
    if train_features:
        sample_feature = train_features[0]
        try:
            feature_names = sample_feature.get_feature_names()
            print(f"✓ Got feature names from sample feature: {len(feature_names)} features")
        except AttributeError:
            # Method 3: Extract from feature vector attributes
            if hasattr(sample_feature, '__dict__'):
                feature_names = [attr for attr in sample_feature.__dict__.keys() 
                               if not attr.startswith('_') and attr not in ['condition_label', 'sample_id']]
                print(f"✓ Extracted feature names from attributes: {len(feature_names)} features")
            else:
                # Method 4: Fallback to generic names
                n_features = len(sample_feature.to_array()) if hasattr(sample_feature, 'to_array') else sklearn_model.n_features_in_
                feature_names = [f'feature_{i}' for i in range(n_features)]
                print(f"✓ Using generic feature names: {len(feature_names)} features")

# Get class names
class_names = eval_metrics['classes']

### 5.1 Visual Tree Plot
plt.figure(figsize=(20, 12))
plot_tree(sklearn_model, 
          feature_names=feature_names,
          class_names=class_names,
          filled=True,
          rounded=True,
          fontsize=12,
          max_depth=5,  # Limit depth for readability
)
plt.title('Decision Tree Structure (Top 5 Levels)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Classification with Random Forest

In [ ]:
from ambient.classification.rf_classifier import (
    RFClassifierConfig,
    RFGaitClassifier
)

In [ ]:
rf_config = RFClassifierConfig(
    n_estimators = 100,
    max_depth = 5,
)

rf_classifier = RFGaitClassifier(rf_config)

### Training

In [ ]:
rf_metrics = rf_classifier.train(
    features=train_features,
    validate=True,
    auto_remove_invalid=True
)

print(f"training accuracy: {rf_metrics['train_accuracy']}")

### Evaluation 

In [ ]:
eval_metrics = rf_classifier.evaluate(train_features)
print(f"Eval accuracy: {eval_metrics['accuracy']}")

In [ ]:
eval_metrics = rf_classifier.evaluate(test_features)
print(f"Eval accuracy: {eval_metrics['accuracy']}")

In [ ]:
report = eval_metrics['classification_report']
for class_name in eval_metrics['classes']:
    cm = report[class_name]
    print(f"\n{class_name}")
    print(f"  F1-Score : {cm['f1-score']:.3f}")
    print(f"  Precision: {cm['precision']:.3f}")
    print(f"  Recall   : {cm['recall']:.3f}")    

### Evaluation Confusion Matrix

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(
    eval_metrics['confusion_matrix'],
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=eval_metrics['classes'],
    yticklabels=eval_metrics['classes']
)
plt.title('Random Forest Classifier - Confusion Matrix', fontsize=15)
plt.ylabel('True Class Label', fontsize=12)
plt.xlabel('Precited Class Label', fontsize=12)
plt.tight_layout()
plt.show()